# Demo code for PCA lecture

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ All libraries imported successfully!")

In [ ]:
def generate_bearing_data(n_samples=500):
    """
    Generate synthetic bearing data with realistic measurements
    - 250 healthy bearings (lower vibration, normal temperature)
    - 250 faulty bearings (higher vibration, elevated temperature)
    """
    np.random.seed(42)
    n_healthy = n_samples // 2
    n_faulty = n_samples - n_healthy
    
    # Healthy bearings - normal operation
    healthy_temp = np.random.normal(65, 5, n_healthy)              # Temperature (°C) - Bearing housing temperature
    healthy_vib_x = np.random.normal(1.5, 0.4, n_healthy)          # X-axis vibration (mm/s)
    healthy_vib_y = np.random.normal(1.6, 0.4, n_healthy)          # Y-axis vibration (mm/s)
    healthy_vib_z = np.random.normal(1.4, 0.4, n_healthy)          # Z-axis vibration (mm/s)
    healthy_vib_avg = (healthy_vib_x + healthy_vib_y + healthy_vib_z) / 3
    healthy_vib_max = np.maximum(np.maximum(healthy_vib_x, healthy_vib_y), healthy_vib_z)
    healthy_vib_range = healthy_vib_max - np.minimum(np.minimum(healthy_vib_x, healthy_vib_y), healthy_vib_z)
    healthy_imbalance = np.random.normal(0.02, 0.005, n_healthy)   # Shaft imbalance (mm)
    healthy_temp_2 = healthy_temp * 0.98 + np.random.normal(0, 1, n_healthy)  # Correlated temp sensor - Bearing outer race temperature
    healthy_vib_rms = np.sqrt((healthy_vib_x**2 + healthy_vib_y**2 + healthy_vib_z**2) / 3)
    healthy_speed = np.random.normal(1500, 50, n_healthy)          # RPM
    healthy_load = np.random.normal(0.6, 0.1, n_healthy)           # Load factor (0-1)
    
    # Faulty bearings - degraded operation
    faulty_temp = np.random.normal(85, 8, n_faulty)                # Higher temperature
    faulty_vib_x = np.random.normal(5.0, 1.2, n_faulty)            # Higher X vibration
    faulty_vib_y = np.random.normal(5.5, 1.3, n_faulty)            # Higher Y vibration
    faulty_vib_z = np.random.normal(4.8, 1.1, n_faulty)            # Higher Z vibration
    faulty_vib_avg = (faulty_vib_x + faulty_vib_y + faulty_vib_z) / 3
    faulty_vib_max = np.maximum(np.maximum(faulty_vib_x, faulty_vib_y), faulty_vib_z)
    faulty_vib_range = faulty_vib_max - np.minimum(np.minimum(faulty_vib_x, faulty_vib_y), faulty_vib_z)
    faulty_imbalance = np.random.normal(0.08, 0.02, n_faulty)      # Higher imbalance
    faulty_temp_2 = faulty_temp * 0.98 + np.random.normal(0, 1, n_faulty)
    faulty_vib_rms = np.sqrt((faulty_vib_x**2 + faulty_vib_y**2 + faulty_vib_z**2) / 3)
    faulty_speed = np.random.normal(1480, 60, n_faulty)            # Slightly lower RPM
    faulty_load = np.random.normal(0.7, 0.15, n_faulty)            # Higher load
    
    # Combine all features
    healthy_data = np.column_stack([
        healthy_temp, healthy_vib_x, healthy_vib_y, healthy_vib_z,
        healthy_vib_avg, healthy_vib_max, healthy_vib_range,
        healthy_imbalance, healthy_temp_2, healthy_vib_rms,
        healthy_speed, healthy_load
    ])
    
    faulty_data = np.column_stack([
        faulty_temp, faulty_vib_x, faulty_vib_y, faulty_vib_z,
        faulty_vib_avg, faulty_vib_max, faulty_vib_range,
        faulty_imbalance, faulty_temp_2, faulty_vib_rms,
        faulty_speed, faulty_load
    ])
    
    # Create DataFrame
    data = np.vstack([healthy_data, faulty_data])
    labels = np.hstack([np.zeros(n_healthy), np.ones(n_faulty)])
    
    feature_names = [
        'temperature', 'vib_x', 'vib_y', 'vib_z',
        'vib_avg', 'vib_max', 'vib_range',
        'imbalance', 'temperature_2', 'vib_rms',
        'speed_rpm', 'load_factor'
    ]
    
    df = pd.DataFrame(data, columns=feature_names)
    df['fault_status'] = labels.astype(int)
    
    return df

df = generate_bearing_data(n_samples=500)
print("✓ Dataset generated successfully!")
print(f"  Shape: {df.shape}")
print(f"  Features: {df.shape[1] - 1}")
print(f"  Samples: {df.shape[0]}")
print(f"  Healthy bearings: {(df['fault_status']==0).sum()}")
print(f"  Faulty bearings: {(df['fault_status']==1).sum()}")

## SECTION 1: Load and Explore Data
**→ Corresponds to Slide: "Running Example: Bearing Fault Detection"**

In [ ]:

print("First 5 bearings in our dataset:")
display(df.head())
print("\n" + "="*80 + "\n")


print("Dataset Information:")
print(f"Total samples: {len(df)}")
print(f"Number of features: {len(df.columns) - 1}")
print(f"Healthy bearings: {(df['fault_status']==0).sum()}")
print(f"Faulty bearings: {(df['fault_status']==1).sum()}")
print("\n" + "="*80 + "\n")


print("Summary Statistics:")
display(df.describe())

## SECTION 2: Visualize Features
**→ Corresponds to Slide: "Features: The Data We Measure"**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features_to_plot = ['temperature', 'vib_x', 'vib_y', 'vib_z', 'vib_avg', 'imbalance']

for idx, feature in enumerate(features_to_plot):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    
    healthy = df[df['fault_status'] == 0][feature]
    faulty = df[df['fault_status'] == 1][feature]
    
    ax.hist(healthy, bins=25, alpha=0.6, label='Healthy', color='green', edgecolor='black')
    ax.hist(faulty, bins=25, alpha=0.6, label='Faulty', color='red', edgecolor='black')
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {feature}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Feature Distributions: Healthy vs Faulty Bearings', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\nObservations:")
print("• Faulty bearings have higher temperatures (85°C vs 65°C)")
print("• Faulty bearings vibrate much more (5+ mm/s vs 1.5 mm/s)")
print("• Faulty bearings have more imbalance (0.08 mm vs 0.02 mm)")

In [ ]:
feature_cols = df.columns[:-1]  # All except fault_status
ranges = df[feature_cols].max() - df[feature_cols].min()
means = df[feature_cols].mean()

scale_comparison = pd.DataFrame({
    'Feature': feature_cols,
    'Min': df[feature_cols].min(),
    'Max': df[feature_cols].max(),
    'Range': ranges,
    'Mean': means
})

print("\nFeature Scales and Ranges:")
print(scale_comparison.sort_values('Range', ascending=False).to_string(index=False))
print("\n⚠ Notice: temperature range is ~50, while imbalance range is ~0.13")
print("   This difference will cause problems without scaling!")

## SECTION 3: Split the Data
**→ Corresponds to Slide: "Training, Validation, and Testing"**

In [ ]:
# features and target
X = df.drop('fault_status', axis=1)
y = df['fault_status']

print("Separating features (X) and target (y):")
print(f"  X shape: {X.shape} - the measurements")
print(f"  y shape: {y.shape} - the labels (0=healthy, 1=faulty)")
print("\n" + "="*80 + "\n")

# Split into train and test sets (for simplicity, we'll use train for both train and validation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30,      # 30% for testing
    random_state=42,
    stratify=y           # Keep same proportion of healthy/faulty in both sets
)

print("Data Split:")
print(f"  Training set: {X_train.shape[0]} bearings ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"  Test set: {X_test.shape[0]} bearings ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"\nTraining set composition:")
print(f"  Healthy: {(y_train==0).sum()}")
print(f"  Faulty: {(y_train==1).sum()}")
print(f"\nTest set composition:")
print(f"  Healthy: {(y_test==0).sum()}")
print(f"  Faulty: {(y_test==1).sum()}")

## SECTION 4: Apply Feature Scaling
**→ Corresponds to Slide: "Feature Scaling: Two Common Methods"**

In [ ]:
# Method 1: Min-Max Normalization (scale to 0-1)
print("METHOD 1: Min-Max Normalization")
print("="*80)

minmax_scaler = MinMaxScaler()
minmax_scaler.fit(X_train)  
X_train_minmax = minmax_scaler.transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("After Min-Max scaling (first 5 features):")
print(f"{'Feature':<20} {'Original Min':<15} {'Original Max':<15} {'Scaled Min':<15} {'Scaled Max':<15}")
print("-" * 80)
for i in range(5):
    feat = X.columns[i]
    orig_min = X_train.iloc[:, i].min()
    orig_max = X_train.iloc[:, i].max()
    scaled_min = X_train_minmax[:, i].min()
    scaled_max = X_train_minmax[:, i].max()
    print(f"{feat:<20} {orig_min:<15.2f} {orig_max:<15.2f} {scaled_min:<15.4f} {scaled_max:<15.4f}")

print("\n✓ All features now in range [0, 1]\n")

# Method 2: Standardization (mean=0, std=1)
print("METHOD 2: Standardization (Z-score)")
print("="*80)

standard_scaler = StandardScaler()
standard_scaler.fit(X_train)  
X_train_scaled = standard_scaler.transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

print("After Standardization (first 5 features):")
print(f"{'Feature':<20} {'Original Mean':<15} {'Original Std':<15} {'Scaled Mean':<15} {'Scaled Std':<15}")
print("-" * 80)
for i in range(5):
    feat = X.columns[i]
    orig_mean = X_train.iloc[:, i].mean()
    orig_std = X_train.iloc[:, i].std()
    scaled_mean = X_train_scaled[:, i].mean()
    scaled_std = X_train_scaled[:, i].std()
    print(f"{feat:<20} {orig_mean:<15.2f} {orig_std:<15.2f} {scaled_mean:<15.6f} {scaled_std:<15.4f}")

print("\n✓ All features now have mean≈0 and std≈1")
print("\nWe'll use STANDARDIZATION for the rest of the demo (better for PCA)")

## SECTION 5: Compare Scaled vs Unscaled
**→ Corresponds to Slide: "Visualizing the Impact of Scaling"**

In [ ]:
feature_to_show = 'vib_x'
feature_idx = list(X.columns).index(feature_to_show)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(X_train[feature_to_show], bins=30, edgecolor='black', color='skyblue')
axes[0].set_xlabel(feature_to_show)
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Original Data\n(Range: {X_train[feature_to_show].min():.2f} to {X_train[feature_to_show].max():.2f})')
axes[0].grid(True, alpha=0.3)

axes[1].hist(X_train_minmax[:, feature_idx], bins=30, edgecolor='black', color='lightgreen')
axes[1].set_xlabel(f'{feature_to_show} (scaled)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('After Min-Max Scaling\n(Range: 0 to 1)')
axes[1].grid(True, alpha=0.3)

axes[2].hist(X_train_scaled[:, feature_idx], bins=30, edgecolor='black', color='lightcoral')
axes[2].set_xlabel(f'{feature_to_show} (scaled)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('After Standardization\n(Mean≈0, Std≈1)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nIMPACT ON DISTANCE CALCULATIONS:")
print("="*80)
sample1_raw = X_train.iloc[0].values
sample2_raw = X_train.iloc[1].values
distance_raw = np.linalg.norm(sample1_raw - sample2_raw)

sample1_scaled = X_train_scaled[0]
sample2_scaled = X_train_scaled[1]
distance_scaled = np.linalg.norm(sample1_scaled - sample2_scaled)

print(f"Distance between first two samples:")
print(f"  Without scaling: {distance_raw:.4f}")
print(f"  With scaling:    {distance_scaled:.4f}")

feature_diffs = np.abs(sample1_raw - sample2_raw)
top_contributors = pd.DataFrame({
    'Feature': X.columns,
    'Difference': feature_diffs,
    'Contribution %': (feature_diffs / distance_raw) * 100
}).sort_values('Contribution %', ascending=False)

print("\nFeatures dominating the UNSCALED distance:")
print(top_contributors.head(3).to_string(index=False))
print("\nNotice: Large-scale features dominate, even if less important!")

## SECTION 6: Apply PCA
**→ Corresponds to Slide: "The Solution: Principal Component Analysis (PCA)"**

In [ ]:
print("APPLYING PCA")
print("="*80)

# pply PCA with all components to see the full picture
pca_full = PCA()
pca_full.fit(X_train_scaled)

print(f"Original number of features: {X_train_scaled.shape[1]}")
print(f"Number of principal components: {pca_full.n_components_}")
print("\nExplained variance by each component:")
print(f"{'Component':<15} {'Variance %':<15} {'Cumulative %':<15}")
print("-" * 45)

cumsum = 0
for i, var in enumerate(pca_full.explained_variance_ratio_, 1):
    cumsum += var
    print(f"PC{i:<14} {var*100:<15.2f} {cumsum*100:<15.2f}")

## SECTION 7: Analyze Explained Variance
**→ Corresponds to Slide: "How Much Information Do We Keep?"**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

explained_var = pca_full.explained_variance_ratio_
ax1.plot(range(1, len(explained_var)+1), explained_var, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Variance per Component')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, len(explained_var)+1))

cumsum_var = np.cumsum(explained_var)
ax2.plot(range(1, len(cumsum_var)+1), cumsum_var, 'ro-', linewidth=2, markersize=8)
ax2.axhline(y=0.90, color='green', linestyle='--', linewidth=2, label='90% threshold')
ax2.axhline(y=0.95, color='blue', linestyle='--', linewidth=2, label='95% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, len(cumsum_var)+1))
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

n_components_90 = np.argmax(cumsum_var >= 0.90) + 1
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1

print(f"\nHow many components do we need?")
print(f"  For 90% of variance: {n_components_90} components")
print(f"  For 95% of variance: {n_components_95} components")
print(f"\nReduction ratio (95% threshold): {X_train_scaled.shape[1]}→{n_components_95} " +
      f"({X_train_scaled.shape[1]/n_components_95:.1f}:1)")

n_components = n_components_95
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"\nFinal PCA transformation:")
print(f"  Original shape: {X_train_scaled.shape}")
print(f"  Reduced shape:  {X_train_pca.shape}")
print(f"  Variance retained: {cumsum_var[n_components-1]*100:.1f}%")

## SECTION 8: Visualize PCA Results
**→ Corresponds to Slide: "Visualizing the Data After PCA"**

In [ ]:
plt.figure(figsize=(10, 8))

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

scatter = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1],
                     c=y_train, cmap='RdYlGn_r',
                     alpha=0.6, edgecolors='black', s=50)

plt.xlabel(f'PC1 ({pc1_var:.1f}% of variance)', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({pc2_var:.1f}% of variance)', fontsize=12, fontweight='bold')
plt.title('Bearing Data in Principal Component Space', fontsize=14, fontweight='bold')

cbar = plt.colorbar(scatter, label='Bearing Status')
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Healthy', 'Faulty'])

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print("="*80)
print("• Horizontal axis (PC1): Captures the most variation - likely overall bearing health")
print("• Vertical axis (PC2): Captures second most variation")
print("• Green points (left): Healthy bearings")
print("• Red points (right): Faulty bearings")
print("• Notice: Good separation between healthy and faulty bearings!")
print(f"\nThis 2D plot represents {pc1_var+pc2_var:.2f}% of the information from 12 original features!")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
pc3_var = pca.explained_variance_ratio_[2] * 100

healthy_mask = y_train == 0
faulty_mask = y_train == 1

ax.scatter(X_train_pca[healthy_mask, 0],
          X_train_pca[healthy_mask, 1],
          X_train_pca[healthy_mask, 2],
          c='green', label='Healthy', alpha=0.6, s=30, edgecolors='black')

ax.scatter(X_train_pca[faulty_mask, 0],
          X_train_pca[faulty_mask, 1],
          X_train_pca[faulty_mask, 2],
          c='red', label='Faulty', alpha=0.6, s=30, edgecolors='black')

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontsize=11, fontweight='bold')
ax.set_zlabel(f'PC3 ({pc3_var:.1f}%)', fontsize=11, fontweight='bold')
ax.set_title('3D PCA Visualization', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n3D visualization shows {pc1_var+pc2_var+pc3_var:.1f}% of total variance")

## SECTION 9: Interpret Component Loadings
**→ Corresponds to Slide: "What Do Components Mean Physically?"**

In [ ]:
print("COMPONENT INTERPRETATION")
print("="*80)
print("\nPrincipal components are combinations of original features.")
print("The 'weights' show how much each feature contributes.\n")

components_df = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=X.columns
)

print("Component Weights (first 3 components):")
print(components_df.iloc[:, :3].round(3))

print("\n" + "="*80)
print("\nMost important features for each component:")
print("-"*80)

for i in range(min(3, pca.n_components_)):
    print(f"\nPC{i+1} (explains {pca.explained_variance_ratio_[i]*100:.1f}% of variance):")
    
    loadings = components_df.iloc[:, i].abs().sort_values(ascending=False)
    
    print("  Top contributors:")
    for j, (feat, loading) in enumerate(loadings.head(4).items(), 1):
        actual_loading = components_df.loc[feat, f'PC{i+1}']
        print(f"    {j}. {feat:<20} (weight: {actual_loading:>6.3f})")

plt.figure(figsize=(10, 8))
n_pcs_to_show = min(5, pca.n_components_)
sns.heatmap(components_df.iloc[:, :n_pcs_to_show].T,
           cmap='RdBu_r', center=0, annot=True, fmt='.2f',
           cbar_kws={'label': 'Weight Value'},
           linewidths=0.5)
plt.title('PCA Component Weights\n(How original features combine into components)',
         fontsize=13, fontweight='bold')
plt.xlabel('Original Features', fontsize=11, fontweight='bold')
plt.ylabel('Principal Components', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("INTERPRETATION GUIDE:")
print("-"*80)
print("• Positive weights: Feature increases → Component increases")
print("• Negative weights: Feature increases → Component decreases")
print("• Large absolute values: Feature is important for this component")
print("• Small absolute values: Feature contributes little to this component")

## SECTION 10: Complete Pipeline and Results
**→ Corresponds to Slide: "Putting It All Together" & "Results: Before and After PCA"**

In [ ]:
import time

print("COMPLETE MACHINE LEARNING PIPELINE")
print("="*80)
print("\nWe'll train a simple classifier to predict bearing failures")
print("Compare performance using ALL features vs PCA-reduced features\n")

# Model 1: Using all original features (scaled)
print("MODEL 1: Using All 12 Features")
print("-"*80)
start_time = time.time()
clf_full = LogisticRegression(random_state=42, max_iter=1000)
clf_full.fit(X_train_scaled, y_train)
time_full = time.time() - start_time

y_pred_full = clf_full.predict(X_test_scaled)
acc_full = accuracy_score(y_test, y_pred_full)

print(f"Training time: {time_full:.4f} seconds")
print(f"Test accuracy: {acc_full:.4f} ({acc_full*100:.1f}%)")

print("\nDetailed Results:")
print(classification_report(y_test, y_pred_full, target_names=['Healthy', 'Faulty']))

# Model 2: Using PCA-reduced features
print("\n" + "="*80)
print(f"MODEL 2: Using {pca.n_components_} Principal Components")
print("-"*80)
start_time = time.time()
clf_pca = LogisticRegression(random_state=42, max_iter=1000)
clf_pca.fit(X_train_pca, y_train)
time_pca = time.time() - start_time

y_pred_pca = clf_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"Training time: {time_pca:.4f} seconds")
print(f"Test accuracy: {acc_pca:.4f} ({acc_pca*100:.1f}%)")

print("\nDetailed Results:")
print(classification_report(y_test, y_pred_pca, target_names=['Healthy', 'Faulty']))

print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)
print(f"{'Metric':<30} {'All Features':<20} {'PCA Features':<20}")
print("-" * 70)
print(f"{'Number of features':<30} {X_train_scaled.shape[1]:<20} {pca.n_components_:<20}")
print(f"{'Information retained':<30} {'100%':<20} {f'{cumsum_var[pca.n_components_-1]*100:.1f}%':<20}")
print(f"{'Training time (seconds)':<30} {time_full:<20.4f} {time_pca:<20.4f}")
print(f"{'Test accuracy':<30} {f'{acc_full:.4f} ({acc_full*100:.1f}%)':<20} {f'{acc_pca:.4f} ({acc_pca*100:.1f}%)':<20}")
print(f"{'Speedup':<30} {'1.0x':<20} {f'{time_full/time_pca:.2f}x':<20}")

print("\n✓ Similar accuracy with fewer features and faster training!")
print("✓ PCA successfully reduced complexity while maintaining performance")

## BONUS: Information Loss from PCA

**→ Optional demonstration showing what information is lost**

In [ ]:
# reconstruction to understand information loss
print("UNDERSTANDING INFORMATION LOSS")
print("="*80)
print("\nPCA is 'lossy' - we lose some information in the reduction.")
print("Let's see what happens when we reconstruct the original data:\n")

# original features from PCA components
X_test_reconstructed = pca.inverse_transform(X_test_pca)

# reconstruction error
mse = np.mean((X_test_scaled - X_test_reconstructed) ** 2)
print(f"Mean Squared Reconstruction Error: {mse:.6f}")

# per-feature reconstruction error
feature_mse = np.mean((X_test_scaled - X_test_reconstructed) ** 2, axis=0)
reconstruction_errors = pd.DataFrame({
    'Feature': X.columns,
    'Reconstruction Error': feature_mse
}).sort_values('Reconstruction Error', ascending=False)

print("\nReconstruction errors by feature:")
print(reconstruction_errors.to_string(index=False))

# reconstruction for one sample
sample_idx = 0
original = X_test_scaled[sample_idx]
reconstructed = X_test_reconstructed[sample_idx]

plt.figure(figsize=(12, 6))
x_pos = np.arange(len(X.columns))
width = 0.35

plt.bar(x_pos - width/2, original, width, label='Original', alpha=0.8, color='skyblue')
plt.bar(x_pos + width/2, reconstructed, width, label='Reconstructed', alpha=0.8, color='lightcoral')

plt.xlabel('Features', fontsize=11, fontweight='bold')
plt.ylabel('Standardized Value', fontsize=11, fontweight='bold')
plt.title(f'Original vs Reconstructed Features (Test Sample {sample_idx})\n' +
         f'Retained {cumsum_var[pca.n_components_-1]*100:.1f}% of variance',
         fontsize=12, fontweight='bold')
plt.xticks(x_pos, X.columns, rotation=45, ha='right')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY INSIGHT:")
print("-"*80)
print(f"Even though we kept only {pca.n_components_} out of {X_train_scaled.shape[1]} components,")
print(f"the reconstruction is very close to the original!")
print(f"This confirms we retained {cumsum_var[pca.n_components_-1]*100:.1f}% of the important information.")